# CLAUDE API MECHANICS

<br>

**About:** Every core Claude API surface: sync and streaming calls, tools, vision, thinking, and the Message Batches API.

**Learning Goals:**
- Make Messages API calls (sync, streaming, vision, with tools) using the anthropic SDK.
- Recognize when to use the Message Batches API and how to submit, poll, and retrieve results.
- Interpret every stop_reason value the API can emit.
- Handle transient errors with backoff without silently retrying non-retryable ones.
- Estimate token counts and cost for a given prompt using count_tokens.

**Keywords:** messages, streaming, batches, stop_reason, prompt caching, count_tokens

**Prerequisite Knowledge:** (1) HTTP + JSON basics, (2) Python 3.11+ context managers

**Target User:** Developer with Python fluency who has an ANTHROPIC_API_KEY


**Estimated Cost to Run:** $0.03 default; $0.15 with live batches run

**Estimated Completion:** 45-60 min

<hr style="border: 4px solid#003262;" />



#### CONTENTS

- [Part 1: The Messages API - One Round Trip](#part-1-the-messages-api-one-round-trip)
- [Part 2: Streaming Responses](#part-2-streaming-responses)
- [Part 3: Tools, tool_use, tool_result](#part-3-tools-tooluse-toolresult)
- [Part 4: Vision and Multi-format Input](#part-4-vision-and-multi-format-input)
- [Part 4b: Extended Thinking](#part-4b-extended-thinking-introductory)
- [Part 5: Batches API and the Realtime-vs-Batch Decision](#part-5-batches-api-and-the-realtime-vs-batch-decision)

<br>

In [1]:
# Paths in this repo are written relative to the repository root, but a
# Jupyter kernel starts in the notebook's own directory. Anchor to the root.
from claude_code.paths import use_repo_root
use_repo_root()

# Setup - imports and environment check
import os
import sys
import json
import base64
import subprocess
from pathlib import Path

import anthropic  # Verified against pypi.org/pypi/anthropic/json on 2026-09-02

from claude_code.api import MessagesClient, StreamingClient, BatchesClient, BatchRequest, CallResult
from claude_code.api.errors import BudgetExceededError
from claude_code.config.model_registry import MODELS, get_model, cost_for_call
from claude_code.security.secrets import load_api_key
from claude_code.tools import Tool, Dispatcher

DRY_RUN = os.getenv("CLAUDE_CODE_DRY_RUN", "0") == "1"
if DRY_RUN:
    print("DRY RUN: live API calls will be skipped")
    # Client objects need a key to construct; no request is ever sent in dry-run mode.
    os.environ.setdefault("ANTHROPIC_API_KEY", "dry-run-placeholder")
else:
    _ = load_api_key("ANTHROPIC_API_KEY")  # raises MissingSecretError with helpful message if unset

print(f"anthropic SDK: {anthropic.__version__}")
print(f"Available model keys: {list(MODELS.keys())}")

anthropic SDK: 1.3.0
Available model keys: ['fable_current', 'opus_current', 'sonnet_current', 'haiku_current']


<hr style="border: 2px solid#003262;" />

## PART 1: The **Messages API** - One Round Trip

```mermaid
flowchart LR
    R["model, max_tokens,<br/>system, messages"] --> API["messages.create"]
    API --> RESP["response"]
    RESP --> C["content:<br/>a list of typed blocks,<br/>never a bare string"]
    RESP --> S["stop_reason:<br/>why generation ended"]
    RESP --> U["usage:<br/>input and output tokens"]
```

Every call to Claude follows the same request/response shape: you send a `messages` list to `client.messages.create()` and receive a **response object**. The key fields to know for the exam are:


| Field | Type | What it means |
|---|---|---|
| `id` | str | Unique ID for this response |
| `content` | list | Always a list of content blocks - never a bare string |
| `stop_reason` | str | Why generation stopped: `end_turn`, `tool_use`, `max_tokens`, `stop_sequence` |
| `usage` | object | `input_tokens`, `output_tokens` billed |
| `model` | str | The model ID that served this response |
| `role` | str | Always `assistant` |
| `type` | str | Always `message` |

The `content` field is a list of typed blocks. A simple text response contains exactly one block with `type: "text"`. Tool-use responses may contain `tool_use` blocks alongside or instead of text blocks.

In [2]:
# Tier 1 - raw Messages API call, inspect every response field
if not DRY_RUN:
    raw_client = anthropic.Anthropic(api_key=load_api_key())

    # Verified against platform.claude.com/docs/en/api/messages on 2026-09-02
    resp = raw_client.messages.create(
        model="claude-haiku-4-5-20251001",  # Verified against platform.claude.com/docs/en/docs/about-claude/models on 2026-09-02
        max_tokens=64,
        messages=[{"role": "user", "content": "What color is the sky? One word only."}],
    )
    print("id:          ", resp.id)
    print("model:       ", resp.model)
    print("role:        ", resp.role)
    print("type:        ", resp.type)
    print("stop_reason: ", resp.stop_reason)
    print("usage:       ", resp.usage)
    print("content:     ", resp.content)
    print("content[0].type:", resp.content[0].type)
    print("content[0].text:", resp.content[0].text)
else:
    print("DRY RUN: skipping live API call (Part 1 Tier 1)")

id:           msg_011CfWVxiUdbKtsU2ifyXMG8
model:        claude-haiku-4-5-20251001
role:         assistant
type:         message
stop_reason:  end_turn
usage:        Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=17, output_tokens=5, output_tokens_details=None, server_tool_use=None, service_tier='standard')
content:      [TextBlock(citations=None, text='Blue.', type='text')]
content[0].type: text
content[0].text: Blue.


Notice that `content` is a list even for a simple text response. This matters in code: never do `resp.content` directly - iterate or index it. The next tier uses `MessagesClient` from `claude_code.api` which normalizes this into a `CallResult` with a `.text` shortcut.

In [4]:
# Verified against https://docs.anthropic.com/en/api/messages 2026-09-08
# Tier 2 - same call via MessagesClient; CallResult bundles text, cost, and usage
if not DRY_RUN:
    with MessagesClient(model="haiku_current", budget_usd=0.05) as mc:
        result = mc.call(
            messages=[{"role": "user", "content": "What color is the sky? One word only."}],
            max_tokens=64,
        )
    print("text:          ", result.text)
    print("stop_reason:   ", result.stop_reason)
    print("input_tokens:  ", result.input_tokens)
    print("output_tokens: ", result.output_tokens)
    print(f"cost_usd:       ${result.cost_usd:.6f}")
    print("model_id:      ", result.model_id)
else:
    print("DRY RUN: skipping live API call (Part 1 Tier 2)")

text:           Blue.
stop_reason:    end_turn
input_tokens:   17
output_tokens:  5
cost_usd:       $0.000042
model_id:       claude-haiku-4-5-20251001


`MessagesClient` also handles retries (429, 5xx) and budget enforcement. The `Tier 3` script for this Part is the streaming smoke test introduced in Part 2 - we defer to that rather than duplicating a second script here.

In [5]:
# Tier 3 - streaming_client_smoke_test.py in --dry-run mode to show the script interface
result_p = subprocess.run(
    [
        sys.executable, "scripts/streaming_client_smoke_test.py",
        "--prompt", "What color is the sky?",
        "--dry-run",
        "--json-log",
    ],
    capture_output=True, text=True, check=False
)
print("exit code:", result_p.returncode)
print("stdout:   ", result_p.stdout.strip())

exit code: 0
stdout:    {"dry_run": true, "model": "haiku_current", "prompt_preview": "What color is the sky?", "max_tokens": 256}


<!--Navigate back to table of contents-->
<div align="left" style="text-align: left; background-color:#003262;">
    <span>
        <hr style="border: 8px solid#003262;" />
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:0px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:24px;letter-spacing:0px;line-height:20px;padding:24px 40px;text-align:left;text-decoration:none; align:left"> 
            <strong>CONCEPT</strong> CHECK 
        </a>        
    </span>
</div>
<!-------------------------------------->

> **The Messages API returns `response.content` as a list, not a string.** Given the response for a simple text prompt with no tool use, how many content blocks would you expect, and what type would each have?

(a) Zero blocks - the text is in `response.text` as a shortcut.

(b) One block, `type: "text"`.

(c) Two blocks: one `type: "thinking"` and one `type: "text"`.

(d) Variable - it depends on how long the response is.

<br>

In [14]:
# CONCEPT CHECK: content blocks
# How many blocks and what type for a simple text response with no tool use?

### YOUR CODE HERE ###
answer = "..."  # write 'a', 'b', 'c', or 'd' and add a one-sentence justification

### --> https://academy.claude.com/courses/building-with-the-claude-api/handling-message-blocks

<hr style="border: 2px solid#003262;" />

<!-- ANSWER (for reviewer): (b) is correct. A simple text prompt with no tool use produces exactly one content block of type "text". (a) is wrong: there is no response.text attribute on the raw SDK response - you must index content. (c) is wrong: thinking blocks appear only when extended_thinking is explicitly enabled. (d) is wrong: the block count is determined by the response type (text vs tool_use), not by length. -->

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents"> 
            Table of Contents 
        </a>
    </span>
</div>
<!-------------------------------------->

<hr style="border: 2px solid#003262;" />

## PART 2: **Streaming** Responses

Streaming lets the model's output appear in your application as it is generated, reducing perceived latency for users. The SDK exposes this through a context manager:

```python
with client.messages.stream(...) as stream:
    for chunk in stream.text_stream:
        print(chunk, end="", flush=True)
```


> **Key points:** the `with` block owns the HTTP connection - when the block exits (normally or on exception), the connection is closed. If you iterate `stream.text_stream` outside a `with` block, the connection stays open until the SDK's finalizer runs, which can **exhaust the HTTP connection** pool on long-running servers.

In [11]:
# Tier 1 - raw streaming via anthropic SDK context manager
# Verified against platform.claude.com/docs/en/docs/build-with-claude/streaming on 2026-09-02
if not DRY_RUN:
    raw_client = anthropic.Anthropic(api_key=load_api_key())
    full_text = []
    with raw_client.messages.stream(
        model="claude-haiku-4-5-20251001",
        max_tokens=64,
        messages=[{"role": "user", "content": "Count from 1 to 5, one number per line."}],
    ) as stream:
        for chunk in stream.text_stream:
            full_text.append(chunk)
            print(chunk, end="", flush=True)
        final = stream.get_final_message()
    print(f"\nTotal chunks: {len(full_text)}")
    print(f"Input tokens: {final.usage.input_tokens}, Output tokens: {final.usage.output_tokens}")
else:
    print("DRY RUN: skipping live streaming call (Part 2 Tier 1)")

1
2
3
4
5
Total chunks: 2
Input tokens: 21, Output tokens: 13


The raw SDK exposes the connection management directly. `StreamingClient` from `claude_code.api` wraps this in the same context manager pattern and adds automatic cost accounting after the stream ends.

In [12]:
# Tier 2 - streaming via StreamingClient with automatic cost tracking
if not DRY_RUN:
    with StreamingClient(model="haiku_current", budget_usd=0.05) as sc:
        full_text2 = []
        for chunk in sc.stream(
            "Count from 1 to 5, one number per line.",
            max_tokens=64,
        ):
            full_text2.append(chunk)
            print(chunk, end="", flush=True)
        in_tok, out_tok = sc.last_call_tokens
        print(f"\nCost: ${sc.last_call_usd:.6f}, tokens: {in_tok}/{out_tok}")
else:
    print("DRY RUN: skipping live streaming call (Part 2 Tier 2)")

1
2
3
4
5
Cost: $0.000086, tokens: 21/13


For production use or CI pipelines, run the streaming script directly. The `--json-log` flag emits a machine-readable summary including cost and token counts.

In [ ]:
# Tier 3 - invoke streaming_client_smoke_test.py with --json-log
# In --dry-run here to avoid API cost; remove --dry-run flag to run live
result_p = subprocess.run(
    [
        sys.executable, "scripts/streaming_client_smoke_test.py",
        "--prompt", "Count from 1 to 3, one number per line.",
        "--model", "haiku_current",
        "--json-log",
        "--dry-run",
    ],
    capture_output=True, text=True, check=False
)
print("exit code:", result_p.returncode)
print(json.dumps(json.loads(result_p.stdout), indent=2))

<!--Navigate back to table of contents-->
<div align="left" style="text-align: left; background-color:#003262;">
    <span>
        <hr style="border: 8px solid#003262;" />
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:0px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:24px;letter-spacing:0px;line-height:20px;padding:24px 40px;text-align:left;text-decoration:none; align:left"> 
            <strong>CONCEPT</strong> CHECK 
        </a>        
    </span>
</div>
<!-------------------------------------->

> **The stream context manager closes the underlying HTTP connection on `__exit__`.** What happens if you iterate `stream.text_stream` outside of a `with` block?

(a) The iteration still succeeds, but chunks may arrive out of order since nothing enforces the original sequencing.

(b) A `RuntimeError` is raised immediately, because the connection was already torn down by the SDK.

(c) The SDK silently opens a fresh HTTP connection so the stream keeps delivering chunks without interruption.

(d) The connection stays open until the SDK's finalizer eventually runs, which can exhaust the pool on long-running processes.

<br>

In [15]:
# CONCEPT CHECK: streaming connection lifecycle
# What happens when stream.text_stream is iterated outside a with block?

### YOUR CODE HERE ###
answer = "..."  # write 'a', 'b', 'c', or 'd' and add a one-sentence justification

<hr style="border: 2px solid#003262;" />

<!-- ANSWER (for reviewer): (d) is correct. The stream context manager owns the HTTP connection. When used without a with block, there is no __exit__ to close the connection; it stays open until the garbage collector calls the finalizer, which is non-deterministic. On a long-running Jupyter server or FastAPI app, each such leak consumes a connection from the pool, eventually causing new requests to block or fail. (a) is wrong: no RuntimeError is raised immediately. (b) is wrong: ordering is not affected by context management. (d) is wrong: the SDK does not auto-reconnect. -->

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents"> 
            Table of Contents 
        </a>
    </span>
</div>
<!-------------------------------------->

<hr style="border: 2px solid#003262;" />

## PART 3: **Tools**, tool_use, tool_result

Claude can request that your code call a function on its behalf. The flow is:

1. You send a request with `tools=[...]` describing available callables.
2. Claude responds with `stop_reason == "tool_use"` and a `tool_use` block in `content`.
3. You invoke the function and send back the result as a `tool_result` block.
4. Claude produces its final text response.

The `tool_use` block has: `type: "tool_use"`, `id` (unique to this call), `name` (function name), and `input` (dict of args). Your `tool_result` must include `tool_use_id` matching the `id` exactly.

Deep coverage of the full agent loop lives in a deeper dive beyond these three lessons. This Part shows the minimal shape.

In [16]:
# Tier 1 - one tool-calling round trip with a mock get_weather function
if not DRY_RUN:
    raw_client = anthropic.Anthropic(api_key=load_api_key())

    # Verified against platform.claude.com/docs/en/docs/tool-use on 2026-09-02
    tools_def = [
        {
            "name": "get_weather",
            "description": "Return the current weather for a city.",
            "input_schema": {
                "type": "object",
                "properties": {"location": {"type": "string", "description": "City name."}},
                "required": ["location"],
            },
        }
    ]

    resp1 = raw_client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=256,
        tools=tools_def,
        messages=[{"role": "user", "content": "What's the weather in Paris?"}],
    )
    print("stop_reason:", resp1.stop_reason)
    print("content:    ", resp1.content)

    # Extract the tool_use block and send a fake result
    tool_use_block = next(b for b in resp1.content if b.type == "tool_use")
    print(f"\nClaude wants to call: {tool_use_block.name}({tool_use_block.input})")

    # Simulate the tool running
    fake_weather = f"Partly cloudy, 18C in {tool_use_block.input['location']}"

    resp2 = raw_client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=256,
        tools=tools_def,
        messages=[
            {"role": "user", "content": "What's the weather in Paris?"},
            {"role": "assistant", "content": resp1.content},
            {"role": "user", "content": [{
                "type": "tool_result",
                "tool_use_id": tool_use_block.id,
                "content": fake_weather,
            }]},
        ],
    )
    print("\nFinal response:", resp2.content[0].text)
else:
    print("DRY RUN: skipping live tool-use call (Part 3 Tier 1)")

stop_reason: tool_use
content:     [ToolUseBlock(id='toolu_01JfwoosJgircf7kKf1cUFPi', caller=DirectCaller(type='direct'), input={'location': 'Paris'}, name='get_weather', type='tool_use', toolset_name=None)]

Claude wants to call: get_weather({'location': 'Paris'})

Final response: The weather in Paris is currently **partly cloudy** with a temperature of **18°C** (about 64°F).


`claude_code.tools` (see `claude_code/tools`) provides `Tool` and `Dispatcher` to avoid hand-crafting the tool-definition dicts and `tool_result` blocks. `Dispatcher.dispatch_one` validates the input against the JSON Schema before calling the handler.

In [19]:
# Verified against https://docs.anthropic.com/en/api/messages 2026-09-08
# Tier 2 - same round trip via claude_code.tools.Tool and Dispatcher
def mock_get_weather(location: str) -> str:
    return f"Partly cloudy, 18C in {location}"

# Encapsulate the function into a structured Tool object matching the API schema
weather_tool = Tool(
    name="get_weather",
    description="Return the current weather for a city.",
    input_schema={
        "type": "object",
        "properties": {"location": {"type": "string", "description": "City name."}},
        "required": ["location"],
    },
    handler=mock_get_weather,
)

# Initialize the central registry and register tool(s)
dispatcher = Dispatcher()
dispatcher.register(weather_tool)

if not DRY_RUN:
    raw_client = anthropic.Anthropic(api_key=load_api_key())
    # Turn 1: Send the user query along with the list of available tools
    resp1 = raw_client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=256,
        tools=dispatcher.to_api_list(),
        messages=[{"role": "user", "content": "What's the weather in Tokyo?"}],
    )

    # Check if the model has paused execution to request a tool invocation
    if resp1.stop_reason == "tool_use":

        # Locate the specific content block containing the tool use request
        tool_block = next(b for b in resp1.content if b.type == "tool_use")

        # Execute the tool locally via the dispatcher registry mapping
        tool_result = dispatcher.dispatch_one({
            "type": tool_block.type,
            "name": tool_block.name,
            "id": tool_block.id,
            "input": tool_block.input,
        })
        print("Dispatcher returned:", tool_result)

        # Turn 2: Feed the local execution result back to the model to finalize the response
        resp2 = raw_client.messages.create(
            model="claude-haiku-4-5-20251001",
            max_tokens=256,
            tools=dispatcher.to_api_list(),
            messages=[
                {"role": "user", "content": "What's the weather in Tokyo?"},
                {"role": "assistant", "content": resp1.content},
                {"role": "user", "content": [tool_result]},
            ],
        )
        print("Final response:", resp2.content[0].text)
else:
    # Mock validation block for offline environment testing
    print("DRY RUN: skipping live tool-use call (Part 3 Tier 2)")
print("Dispatcher registered tools:", [t["name"] for t in dispatcher.to_api_list()])

Dispatcher returned: {'type': 'tool_result', 'tool_use_id': 'toolu_01NtMTV5gPUmpCcPG9x9zihM', 'content': 'Partly cloudy, 18C in Tokyo'}
Final response: The weather in Tokyo is **partly cloudy** with a temperature of **18°C** (about 64°F).
Dispatcher registered tools: ['get_weather']


For the full agentic tool-calling loop (multiple tool invocations, loop termination, parallel tool use), see a deeper dive beyond these three lessons. This Part covers only the basic tool_use shape of the Messages API.

In [21]:
# Tier 3 - note: deep tool-use loop coverage is in a deeper dive beyond these three lessons
# Confirm dispatcher.to_api_list() matches the Anthropic API shape
api_list = dispatcher.to_api_list()  # Extract list of configured tool definitions
print("Tool API dict keys:", set(api_list[0].keys()))

# Validate that dict contains only exact keys
assert set(api_list[0].keys()) == {"name", "description", "input_schema"}, \
    "Tool dict must have exactly: name, description, input_schema"
print("Shape check passed")

Tool API dict keys: {'name', 'description', 'input_schema'}
Shape check passed


<!--Navigate back to table of contents-->
<div align="left" style="text-align: left; background-color:#003262;">
    <span>
        <hr style="border: 8px solid#003262;" />
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:0px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:24px;letter-spacing:0px;line-height:20px;padding:24px 40px;text-align:left;text-decoration:none; align:left"> 
            <strong>CONCEPT</strong> CHECK 
        </a>        
    </span>
</div>
<!-------------------------------------->

> **You receive a response with `stop_reason == "tool_use"`.** 
> You must respond with which of the following?

(a) A `tool_result` content block whose `tool_use_id` matches the `tool_use` block's `id`.

(b) The tool's raw output as a plain-text string, placed directly in the next `user` message's `content` field.

(c) A new `user` message whose `content` repeats the previous user message verbatim, prompting Claude to retry the call.

(d) An empty `user` message with `content: ""`, signaling to Claude that the tool call was received and acknowledged.

<br>

In [ ]:
# Verified against https://docs.anthropic.com/en/api/messages 2026-09-08
# CONCEPT CHECK: tool_result shape
# What is the correct response when stop_reason == "tool_use"?

### YOUR CODE HERE ###
answer = "..."  # write 'a', 'b', 'c', or 'd' and add a one-sentence justification

<hr style="border: 2px solid#003262;" />

<!-- ANSWER (for reviewer): (a) is correct. The tool_result block requires type: "tool_result", tool_use_id exactly matching the id from the tool_use block (not the response id), and content with the tool output. (a) is wrong: a plain string user message does not identify which tool call is being answered. (b) is wrong: an empty message gives Claude no tool output to reason about. (c) is wrong: repeating the original message starts a new request, not a tool result. -->

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents"> 
            Table of Contents 
        </a>
    </span>
</div>
<!-------------------------------------->

<hr style="border: 2px solid#003262;" />

## PART 4: **Vision** and Multi-format Input

Claude can process images as part of a message. An image content block has two source types:

- **base64**: include the raw bytes encoded as base64 with the `media_type` declared.
- **url**: provide an HTTPS URL to an accessible image (Claude fetches it).

All current Claude models support vision. Verified against platform.claude.com/docs/en/docs/about-claude/models on 2026-09-02.

Image blocks appear in the `content` list of a user message alongside (or instead of) text blocks. You can mix text and image blocks in a single message.

In [23]:
# Tier 1 - create a tiny test PNG programmatically and send it to Haiku
# (avoids requiring a pre-existing file; any real image works the same way)
import struct
import zlib

# Builds valid 8x8 PNG
def _minimal_png(width: int = 8, height: int = 8) -> bytes:

    # Package std PNG with Big-Endian length and CRC32 checksum
    def chunk(name: bytes, data: bytes) -> bytes:
        return struct.pack(">I", len(data)) + name + data + struct.pack(">I", zlib.crc32(name + data) & 0xFFFFFFFF)
    
    # Std 8-byte file signature
    sig = b"\x89PNG\r\n\x1a\n"

    # IHDR Chunk data (Width, Height, Bit Depth=8, Color Type=2 [Truecolor RGB], Interlace=0, etc.)
    ihdr_data = struct.pack(">IIBBBBB", width, height, 8, 2, 0, 0, 0)
    ihdr = chunk(b"IHDR", ihdr_data)

    # Image rows data generation: Filter byte (0x00) + 3 bytes per pixel (R=0xFF, G=0x00, B=0x00)
    raw = b"".join(b"\x00" + b"\xff\x00\x00" * width for _ in range(height))

    # IDAT Chunk (zlib compressed pixel rows) and IEND EOF Marker chunk
    idat = chunk(b"IDAT", zlib.compress(raw))
    iend = chunk(b"IEND", b"")
    return sig + ihdr + idat + iend

# Generate raw binary PNG data and encode it to a standard ASCII base64 string
png_bytes = _minimal_png()
png_b64 = base64.standard_b64encode(png_bytes).decode()

# Verified against platform.claude.com/docs/en/docs/build-with-claude/vision on 2026-09-02# Structural definition compliant with the Anthropic Messages API specification (verified 2026-09-02)
image_block = {
    "type": "image",
    "source": {
        "type": "base64",
        "media_type": "image/png",
        "data": png_b64,
    },
}

if not DRY_RUN:
    raw_client = anthropic.Anthropic(api_key=load_api_key())
    resp = raw_client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=64,
        messages=[{"role": "user", "content": [
            image_block,
            {"type": "text", "text": "What color is this image? One word."},
        ]}],
    )
    print("Vision response:", resp.content[0].text)
else:
    # Off-line diagnostic layout rendering
    print("Image block shape:", {k: ("..." if k == "data" else v) for k, v in image_block["source"].items()})
    print("DRY RUN: skipping live vision call (Part 4 Tier 1)")

Vision response: Red.


`MessagesClient.call` accepts the same messages list format, so you can pass image blocks directly. The shape is identical whether you use the raw SDK or the wrapper.

In [24]:
# Tier 2 - vision call via MessagesClient
if not DRY_RUN:
    # Initialize a self-closing context manager Client wrapper.
    # Specifying 'budget_usd' enforces strict cost controls over the generation loop.
    with MessagesClient(model="haiku_current", budget_usd=0.05) as mc:
        result = mc.call(
            messages=[{"role": "user", "content": [
                image_block,
                {"type": "text", "text": "What color is this image? One word."},
            ]}],
            max_tokens=64,
        )
    print("text:", result.text)
    print(f"cost: ${result.cost_usd:.6f}")
else:
    print("DRY RUN: skipping live vision call (Part 4 Tier 2)")

text: Red
cost: $0.000041


<!--Navigate back to table of contents-->
<div align="left" style="text-align: left; background-color:#003262;">
    <span>
        <hr style="border: 8px solid#003262;" />
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:0px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:24px;letter-spacing:0px;line-height:20px;padding:24px 40px;text-align:left;text-decoration:none; align:left"> 
            <strong>CONCEPT</strong> CHECK 
        </a>        
    </span>
</div>
<!-------------------------------------->

> **Given a 500 KB PNG, what is the correct shape of the message content block that passes the image to Claude via the base64 source type?**

(a) `{"type": "file", "source": {"type": "base64", "media_type": "image/png", "data": "<base64>"}}`

(b) `{"type": "image", "data": "<base64>", "media_type": "image/png"}` - fields sit directly on the block, not nested.

(c) `{"type": "image", "source": {"type": "base64", "media_type": "image/png", "data": "<base64>"}}`

(d) `{"type": "image_url", "url": "<data URI>"}` - the OpenAI-style key name and shape.

<br>

In [ ]:
# CONCEPT CHECK: vision content block shape
# What is the correct shape for a base64 image block?

### YOUR CODE HERE ###
answer = "..."  # write 'a', 'b', 'c', or 'd' and add a one-sentence justification

<hr style="border: 2px solid#003262;" />

<!-- ANSWER (for reviewer): (c) is correct. The image block nests the source under a "source" key with sub-keys "type", "media_type", and "data". (a) is wrong: the data and media_type are NOT top-level fields on the block - they are nested under "source". (c) is wrong: the type is "image", not "file". (d) is wrong: "image_url" is the OpenAI naming convention; Anthropic uses "image" with source.type == "url" for URL-based images. -->

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents"> 
            Table of Contents 
        </a>
    </span>
</div>
<!-------------------------------------->

<hr style="border: 2px solid#003262;" />

## PART 4b: **Extended Thinking** - Introductory

Extended thinking is an API-level mode that surfaces the model's reasoning as `thinking` content blocks in the response, alongside the answer `text` blocks. The wire parameter is `thinking` - there is no `extended_thinking` parameter on `messages.create()`. Passing one raises `TypeError: Messages.create() got an unexpected keyword argument`.

The value has two forms, and which one a model accepts depends on its generation (verified against platform.claude.com/docs/en/docs/about-claude/models on 2026-09-02):

| Form | Meaning | Models |
|---|---|---|
| `thinking={"type": "adaptive"}` | Claude decides when and how deeply to think; pair with `output_config={"effort": ...}` to control spend | Sonnet 5, Opus 5, and the 4.6+ family |
| `thinking={"type": "enabled", "budget_tokens": N}` | Think for at most N tokens (N must be < `max_tokens`, minimum 1024) | Older models only - Haiku 4.5, Sonnet 4.5 |

The fixed-budget form is the one most older material still shows, and it is exactly the form that **returns a 400 on `claude-sonnet-5`**: `budget_tokens` was removed on current models in favour of adaptive thinking. Reach for `budget_tokens` only when you are deliberately pinned to a pre-4.6 model.

Deep coverage of adaptive thinking, effort levels, and when to enable appears in a deeper dive beyond these three lessons. This Part is a minimal introduction so the study guide's explicit mention is addressed.

In [25]:
# Tier 1 - adaptive thinking on Sonnet 5
# The parameter is `thinking`, not `extended_thinking`. On Sonnet 5 the only
# 'adaptive' hands execution steering to the model to scale its depth dynamically.
# Setting display to 'summarized' opts into exposing a readable narrative of the thoughts.
THINKING = {
    "type": "adaptive", 
    "display": "summarized"  # Options: 'summarized' or 'omitted' (default)
    } 

if not DRY_RUN:
    sonnet = get_model("sonnet_current")
    raw_client = anthropic.Anthropic(api_key=load_api_key())
    resp = raw_client.messages.create(
        model=sonnet.id,
        max_tokens=4096,
        thinking=THINKING,                   # Configures thinking behavior
        output_config={"effort": "medium"},  # low | medium | high | xhigh | max
        messages=[{"role": "user", "content": "Is 17 a prime number? Explain your reasoning."}],
    )

    # Extract and preview the high-level thought scratchpad text string
    print("stop_reason:", resp.stop_reason)

    for block in resp.content:
        # Check block types explicitly to cleanly segregate text from reasoning
        if block.type == "thinking":
            # Extract and preview the high-level thought scratchpad text string
            print(f"[thinking block]: {block.thinking[:120]}...")
        elif block.type == "text":
            # Output the finalized, user-facing natural language solution
            print(f"[text]: {block.text}")
else:
    # Diagnostic print paths for offline validation environments
    print("DRY RUN: skipping extended thinking call (Part 4b Tier 1)")
    print(f"Adaptive thinking request shape: thinking={THINKING}")
    print("Legacy fixed-budget shape (pre-4.6 models only): thinking={'type': 'enabled', 'budget_tokens': 2048}")

stop_reason: end_turn
[text]: # Is 17 a Prime Number?

**Yes, 17 is a prime number.**

## Reasoning

A prime number is a whole number greater than 1 that has exactly two divisors: 1 and itself. To check if 17 is prime, I need to see if any number other than 1 and 17 divides it evenly.

**Method: Test divisibility by primes up to √17**

Since √17 ≈ 4.12, I only need to check prime numbers up to 4: **2 and 3**

- **Divisible by 2?** No — 17 is odd (doesn't end in 0, 2, 4, 6, 8)
- **Divisible by 3?** No — digit sum is 1 + 7 = 8, which is not divisible by 3

Since 17 is not divisible by any prime number less than or equal to its square root, it has no factors other than 1 and 17 itself.

## Conclusion

The only way to write 17 as a product of two positive integers is:
$$17 = 1 \times 17$$

Therefore, **17 is prime**. ✓


`MessagesClient.call` takes the same value under the keyword `extended_thinking` and forwards it to the API as `thinking`. The wrapper's keyword name is deliberately the human-facing one; the wire name is what the API sees.

In [26]:
# Tier 2 - adaptive thinking via MessagesClient
if not DRY_RUN:
    # Initialize a self-closing context manager Client wrapper.
    # Specifying 'budget_usd' enforces strict cost controls over the generation loop.
    with MessagesClient(model="sonnet_current", budget_usd=0.10) as mc:
        result = mc.call(
            messages=[{"role": "user", "content": "What is 7 * 8? Explain each step."}],
            max_tokens=4096,                         # Generous token headroom for multi-block responses
            extended_thinking={"type": "adaptive"},  # Internally translated and forwarded as native API 'thinking='
        )

    # Access the clean, flattened text output intended for user viewing    
    print("text:", result.text)

    # Extract the architectural structural layout from the content block array list.
    # Uses a safe fallback (.get) to unpack the block 'type' parameter if the block
    # arrives as a raw dictionary rather than an instantiated object.
    print("content block types:", [b.get("type") if isinstance(b, dict) else b for b in result.content])
else:
    # Diagnostic fallback path for offline runtime validation
    print("DRY RUN: skipping extended thinking call (Part 4b Tier 2)")
print("\nSee a deeper dive beyond these three lessons for adaptive thinking, effort levels, and when-to-enable analysis.")

text: # Multiplying 7 × 8

## Step-by-Step Explanation

**Step 1: Understand what multiplication means**
7 × 8 means adding 7 together, 8 times (or equivalently, adding 8 together, 7 times):
7 + 7 + 7 + 7 + 7 + 7 + 7 + 7

**Step 2: Break it into easier chunks**
Instead of adding one at a time, let's split 8 into 5 + 3:
- 7 × 8 = 7 × (5 + 3)
- 7 × 8 = (7 × 5) + (7 × 3)

**Step 3: Solve each smaller piece**
- 7 × 5 = 35
- 7 × 3 = 21

**Step 4: Add the results together**
35 + 21 = 56

## Answer:
**7 × 8 = 56**
content block types: ['text']

See Notebook 05 Part 5 for adaptive thinking, effort levels, and when-to-enable analysis.


<!--Navigate back to table of contents-->
<div align="left" style="text-align: left; background-color:#003262;">
    <span>
        <hr style="border: 8px solid#003262;" />
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:0px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:24px;letter-spacing:0px;line-height:20px;padding:24px 40px;text-align:left;text-decoration:none; align:left"> 
            <strong>CONCEPT</strong> CHECK 
        </a>        
    </span>
</div>
<!-------------------------------------->

> **A developer enables thinking on a `claude-sonnet-5` call with `thinking={"type": "enabled", "budget_tokens": 2048}` and receives a 400 Bad Request.** Which of the following is the most likely root cause?

(a) The `max_tokens` ceiling was set too low to leave any room for the thinking blocks to be generated at all.

(b) Thinking is still a beta-only feature on Sonnet 5, and the call is missing the required `anthropic-beta` header entirely.

(c) `budget_tokens` was removed on current models; Sonnet 5 uses `{"type": "adaptive"}` instead.

(d) The requested `budget_tokens` value exceeds what fits inside Sonnet 5's total context window for this particular request.

<br>

In [ ]:
# CONCEPT CHECK: thinking parameter shape by model generation
# What causes a 400 when passing budget_tokens to claude-sonnet-5?

### YOUR CODE HERE ###
answer = "..."  # write 'a', 'b', 'c', or 'd' and add a one-sentence justification

<hr style="border: 2px solid#003262;" />

<!-- ANSWER (for reviewer): (c) is correct. On Sonnet 5 (and the whole 4.6+ family) the fixed thinking-token budget was removed: thinking={"type": "enabled", "budget_tokens": N} returns a 400, and the supported forms are {"type": "adaptive"} and {"type": "disabled"}, with depth controlled by output_config.effort. The budget_tokens form is still valid on older models such as Haiku 4.5 and Sonnet 4.5 - the direction of this constraint is the opposite of what most dated exam-prep material implies. (b) is wrong: 2048 is trivially within any model's limits, and the 400 is a schema rejection, not a capacity error. (c) is wrong: max_tokens is a separate ceiling, and the request is rejected before any generation happens. (d) is wrong: thinking needs no beta header on current Claude models. Note also that the parameter name itself is `thinking`; passing `extended_thinking=` to messages.create() is a client-side TypeError, not a 400. -->

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents"> 
            Table of Contents 
        </a>
    </span>
</div>
<!-------------------------------------->

<hr style="border: 2px solid#003262;" />

## PART 5: **Batches API** and the Realtime-vs-Batch Decision

The Message Batches API processes requests asynchronously over a window of up to 24 hours, at approximately 50% of the per-token cost of realtime calls. Use it when:

- Volume is high (hundreds to thousands of documents).
- Results are not needed immediately (latency-tolerant).
- You want to minimize cost at the expense of turnaround time.

Do NOT use it when:
- The call is part of an interactive user-facing flow (latency-sensitive).
- You need the result within seconds.

The workflow is: `submit()` returns a batch ID with status `in_progress`. Poll with `retrieve()` until status is `ended`. Then iterate `results()` to fetch the JSONL output.

In [28]:
# Tier 1 - raw Batches API call with 3 tiny requests
# Verified against platform.claude.com/docs/en/api/creating-message-batches on 2026-09-02
if not DRY_RUN:

    # Submit an array of individual, independent generation tasks as a single bulk request.
    # The Batches API processes these concurrently on remote servers at lower cost thresholds.
    raw_client = anthropic.Anthropic(api_key=load_api_key())
    batch = raw_client.messages.batches.create(
        requests=[
            {
                "custom_id": "q1", 
                "params": {
                    "model": "claude-haiku-4-5-20251001", 
                    "max_tokens": 32, 
                    "messages": [{"role": "user", "content": "Say yes."}]
                }
            },
            {
                "custom_id": "q2", 
                "params": {
                    "model": "claude-haiku-4-5-20251001", 
                    "max_tokens": 32, 
                    "messages": [{"role": "user", "content": "Say no."}]
                }
            },
            {
                "custom_id": "q3", 
                "params": {
                    "model": "claude-haiku-4-5-20251001", 
                    "max_tokens": 32, 
                    "messages": [{"role": "user", "content": "Say maybe."}]
                }   
            },
        ]
    )

    # Print the globally unique identifier assigned to this long-running bulk job
    print("Batch id:     ", batch.id)

    # Track execution phase status (e.g., 'canceling', 'in_progress', 'canceling', 'ended')
    print("Status:       ", batch.processing_status)

    # Inspect sub-object details tracking tracking counts ('processing', 'succeeded', 'errored')
    print("Counts:       ", batch.request_counts)
    
    print("(batch is now in_progress; poll with retrieve() to check status)")
    print("(pass flag real=True to batches_submit_and_poll.py to run end-to-end)")
else:
    # Offline diagnostic route for test run mock executions
    print("DRY RUN: skipping live Batches API call (Part 5 Tier 1)")

Batch id:      msgbatch_01K1MQiByAyttRV551ZohtYf
Status:        in_progress
Counts:        MessageBatchRequestCounts(canceled=0, errored=0, expired=0, processing=3, succeeded=0)
(batch is now in_progress; poll with retrieve() to check status)
(pass flag real=True to batches_submit_and_poll.py to run end-to-end)


`BatchesClient` from `claude_code.api` wraps submit, poll, and results in one coherent interface with duplicate `custom_id` detection and configurable poll intervals.

In [29]:
# Tier 2 - Batches API via BatchesClient
# Instantiate a list of clean, type-safe BatchRequest data structures.
# These object representations abstract away nested boilerplate parameters like 'params' and 'model'.
requests = [
    BatchRequest(custom_id="r1", messages=[{"role": "user", "content": "Say yes."}]),
    BatchRequest(custom_id="r2", messages=[{"role": "user", "content": "Say no."}]),
]

print("Request shapes:")
for r in requests:
    # .to_api_dict() serializes the internal object state into the official API JSON dictionary shape
    print(" ", r.to_api_dict())

if not DRY_RUN:
    # Initialize the purpose-built bulk client coordinator
    client = BatchesClient()

    # Dispatch the batch array. The client wrapper manages uploading payloads,
    # registering entries, and returning a unified tracking reference handle.
    handle = client.submit(requests)
    
    print("Submitted batch:", handle.id, "status:", handle.status)
    print("(poll with client.poll_until_complete(handle.id) to wait for results)")
else:
    print("DRY RUN: showing request shape only (Part 5 Tier 2)")

Request shapes:
  {'custom_id': 'r1', 'params': {'model': 'claude-haiku-4-5-20251001', 'max_tokens': 1024, 'messages': [{'role': 'user', 'content': 'Say yes.'}]}}
  {'custom_id': 'r2', 'params': {'model': 'claude-haiku-4-5-20251001', 'max_tokens': 1024, 'messages': [{'role': 'user', 'content': 'Say no.'}]}}
Submitted batch: msgbatch_01R9QGkK1JTkNA3yoifxXmTz status: in_progress
(poll with client.poll_until_complete(handle.id) to wait for results)


For a full end-to-end round trip (submit, poll, retrieve JSONL), use the script below. The `--real` flag is required to actually submit; default is `--dry-run` to avoid unexpected costs.

For a deep dive comparing this Part's three topologies against a fourth - `AsyncAnthropic` concurrent fan-out - with its own offline test suite, see `irl_examples/01_claude_api_mechanics/synchronous_vs_asynchronous/`.

In [ ]:
# Tier 3 - batches_submit_and_poll.py in --dry-run mode
import tempfile

sample_input = json.dumps([
    {"custom_id": "doc-1", "prompt": "Summarize: The sky is blue.", "model": "haiku_current", "max_tokens": 32},
    {"custom_id": "doc-2", "prompt": "Summarize: Water is wet.", "model": "haiku_current", "max_tokens": 32},
])

with tempfile.NamedTemporaryFile(mode="w", suffix=".json", delete=False) as tmp:
    tmp.write(sample_input)
    tmp_path = tmp.name

result_p = subprocess.run(
    [
        sys.executable, "scripts/batches_submit_and_poll.py",
        "--input", tmp_path,
        "--output", os.path.join(tempfile.gettempdir(), "batch_results.jsonl"),
        "--dry-run",
        "--json-log",
    ],
    capture_output=True, text=True, check=False
)
print("exit code:", result_p.returncode)
print(json.dumps(json.loads(result_p.stdout), indent=2))
print("\nTo run live: omit the dry-run flag and pass the real flag instead")

<!--Navigate back to table of contents-->
<div align="left" style="text-align: left; background-color:#003262;">
    <span>
        <hr style="border: 8px solid#003262;" />
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:0px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:24px;letter-spacing:0px;line-height:20px;padding:24px 40px;text-align:left;text-decoration:none; align:left"> 
            <strong>CONCEPT</strong> CHECK 
        </a>        
    </span>
</div>
<!-------------------------------------->

> **You must process 5,000 documents by tomorrow morning to generate a report.** Cost matters; latency does not. Which of the following decisions is best?

(a) Switch to the smallest model, Haiku, to reduce per-token cost while continuing to issue realtime calls one at a time.

(b) Call the Messages API from a pool of 50 parallel threads so all 5,000 documents finish as quickly as possible.

(c) Cap `max_tokens` at 64 for every call to shrink the per-call bill, and keep using the realtime API as before.

(d) Submit all 5,000 documents as one Message Batch and retrieve the results once the batch finishes.

<br>

In [ ]:
# CONCEPT CHECK: realtime vs batch decision
# You need 5000 documents processed by morning. Cost matters; latency does not.

### YOUR CODE HERE ###
answer = "..."  # write 'a', 'b', 'c', or 'd' and add a one-sentence justification

<hr style="border: 2px solid#003262;" />

<!-- ANSWER (for reviewer): (d) is correct. The Batches API is the right choice when volume is high and latency does not matter - it provides approximately 50% cost reduction on all tokens. (a) is wrong: parallel realtime calls pay full per-token cost; parallelism improves throughput but not per-token price. (c) is wrong: reducing max_tokens lowers the cost per call slightly but does not address the structural cost-vs-latency tradeoff; if the response is truncated by max_tokens the result quality degrades. (d) is wrong: switching to Haiku reduces per-token cost but at the cost of output quality; the Batches API offers the same ~50% discount across ALL models, so the correct lever is batching, not model degradation. -->

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents"> 
            Table of Contents 
        </a>
    </span>
</div>
<!-------------------------------------->

## Sources Consulted

- [Anthropic API Getting Started](https://platform.claude.com/docs/en/api/getting-started) - retrieved 2026-09-02; volatile
- [Messages API Reference](https://platform.claude.com/docs/en/api/messages) - retrieved 2026-09-02; volatile
- [Message Batches API](https://platform.claude.com/docs/en/api/creating-message-batches) - retrieved 2026-09-02; volatile
- [Streaming Guide](https://platform.claude.com/docs/en/docs/build-with-claude/streaming) - retrieved 2026-09-02; volatile
- [Vision Guide](https://platform.claude.com/docs/en/docs/build-with-claude/vision) - retrieved 2026-09-02; volatile
- [Tool Use Guide](https://platform.claude.com/docs/en/docs/tool-use) - retrieved 2026-09-02; volatile

Every code cell that names a version-sensitive detail (option name, class, model ID) is annotated inline with the date it was verified against these sources.

<hr style="border: 6px solid#003262;" />